#### Initialize

In [1]:
import sys
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
LEVEL1_BUCKET_PATH = PARENT / "server/out/places_level1"
LEVEL2_BUCKET_PATH = PARENT / "server/out/places_level2"
LEVEL2_BUCKET_PATH.mkdir(parents=True, exist_ok=True)
LEVEL1_BUCKET = [f for f in LEVEL1_BUCKET_PATH.rglob("*.csv") if f.is_file()]
DF_LEVEL1 = pd.concat([pd.read_csv(f) for f in LEVEL1_BUCKET], ignore_index=True)

#### Parse Types

In [2]:
from server.scripts.clean_places_level_2.type_parsing import parse_type, check_takeaway, predict_cuisine_from_name

df_level2 = DF_LEVEL1.copy()
df_level2["predictedType"] = df_level2.apply(predict_cuisine_from_name, axis=1)
df_level2["cuisineType"] = df_level2.apply(parse_type, axis=1)
df_level2["venueType"] = df_level2.apply(check_takeaway, axis=1)

# ── Diagnostics ───────────────────────────────────────────────────────────────
dist = df_level2["cuisineType"].value_counts()
unresolved = (df_level2["cuisineType"] == "Unspecified").sum()
print(f"Unique cuisineTypes : {dist.nunique()}")
print(f"Still 'Unspecified'  : {unresolved} / {len(df_level2)}  ({unresolved/len(df_level2):.1%})")

Unique cuisineTypes : 40
Still 'Unspecified'  : 1756 / 13092  (13.4%)


In [3]:
unspecified = df_level2[df_level2["cuisineType"]=="Unspecified"]
sample = unspecified[["displayName", "primaryType", "primaryTypeDisplayName", "types", "predictedType", "cuisineType"]]
sample.sample(10)["displayName"].to_list()

['Bread & Truffle - Canary Wharf',
 'Pollo Loco Harrow Road',
 'Mate’s Clapham Restaurant',
 'Becave',
 'Mums spice',
 'Hot Wok.',
 "Roslyn's Kitchen & Bar",
 'Nway Oo Pokhara restaurant',
 'Il Bambini Club London',
 'Camden famous food']

#### Export

In [4]:
for seed_id, group in df_level2.groupby("seed_index"):
    save_path = LEVEL2_BUCKET_PATH / f"{seed_id}.csv"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    group.to_csv(save_path, index=False)